In [2]:
import json
import time

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.patches import Rectangle
from sklearn.preprocessing import LabelEncoder
from sklearn.datasets import make_classification
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split

In [3]:
%%time

data = pd.read_csv("data/final.csv")

CPU times: user 29.8 s, sys: 1.58 s, total: 31.4 s
Wall time: 34.5 s


In [4]:
data.head()

,Active Mean,Bwd IAT Mean,Bwd Pkt Len Std,Bwd Pkts/s,Down/Up Ratio,FIN Flag Cnt,Flow Duration,Flow IAT Min,Flow IAT Std,Flow Pkts/s,...,Fwd Seg Size Min,Idle Mean,Init Bwd Win Byts,Init Fwd Win Byts,Pkt Len Min,Pkt Len Std,RST Flag Cnt,TotLen Fwd Pkts,URG Flag Cnt,Label
0,0.0,8569.50000,655.432936,49.510203,0.0,0,141385,1.0,19069.116850,113.166178,...,20,0.0,119,8192,0.0,474.712955,1,553.0,0,Benign
1,0.0,0.00000,0.000000,3558.718861,0.0,0,281,17.0,174.655375,10676.156580,...,20,0.0,0,123,0.0,21.939310,0,38.0,0,Benign
2,0.0,18494.57143,636.314186,53.605123,1.0,0,279824,1.0,24379.448340,92.915547,...,20,0.0,1047,8192,0.0,566.234209,1,1086.0,0,Benign
3,0.0,0.00000,0.000000,0.000000,0.0,0,132,132.0,0.000000,15151.515150,...,20,0.0,-1,256,0.0,0.000000,0,0.0,0,Benign
4,0.0,21082.83333,611.180489,47.442485,1.0,0,274016,1.0,26311.627030,80.287282,...,20,0.0,1047,8192,0.0,497.254764,1,1285.0,0,Benign


In [5]:
data.columns

Index(['Active Mean', 'Bwd IAT Mean', 'Bwd Pkt Len Std', 'Bwd Pkts/s',
       'Down/Up Ratio', 'FIN Flag Cnt', 'Flow Duration', 'Flow IAT Min',
       'Flow IAT Std', 'Flow Pkts/s', 'Fwd Act Data Pkts', 'Fwd Header Len',
       'Fwd IAT Mean', 'Fwd IAT Min', 'Fwd IAT Std', 'Fwd Pkt Len Mean',
       'Fwd Pkt Len Std', 'Fwd Seg Size Min', 'Idle Mean', 'Init Bwd Win Byts',
       'Init Fwd Win Byts', 'Pkt Len Min', 'Pkt Len Std', 'RST Flag Cnt',
       'TotLen Fwd Pkts', 'URG Flag Cnt', 'Label'],
      dtype='str')

In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10819042 entries, 0 to 10819041
Data columns (total 27 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Active Mean        float64
 1   Bwd IAT Mean       float64
 2   Bwd Pkt Len Std    float64
 3   Bwd Pkts/s         float64
 4   Down/Up Ratio      float64
 5   FIN Flag Cnt       int64  
 6   Flow Duration      int64  
 7   Flow IAT Min       float64
 8   Flow IAT Std       float64
 9   Flow Pkts/s        float64
 10  Fwd Act Data Pkts  int64  
 11  Fwd Header Len     int64  
 12  Fwd IAT Mean       float64
 13  Fwd IAT Min        float64
 14  Fwd IAT Std        float64
 15  Fwd Pkt Len Mean   float64
 16  Fwd Pkt Len Std    float64
 17  Fwd Seg Size Min   int64  
 18  Idle Mean          float64
 19  Init Bwd Win Byts  int64  
 20  Init Fwd Win Byts  int64  
 21  Pkt Len Min        float64
 22  Pkt Len Std        float64
 23  RST Flag Cnt       int64  
 24  TotLen Fwd Pkts    float64
 25  URG Flag Cnt       int64  


In [7]:
X = data.drop(['Label'], axis=1)
y = data['Label']

In [8]:
X

,Active Mean,Bwd IAT Mean,Bwd Pkt Len Std,Bwd Pkts/s,Down/Up Ratio,FIN Flag Cnt,Flow Duration,Flow IAT Min,Flow IAT Std,Flow Pkts/s,...,Fwd Pkt Len Std,Fwd Seg Size Min,Idle Mean,Init Bwd Win Byts,Init Fwd Win Byts,Pkt Len Min,Pkt Len Std,RST Flag Cnt,TotLen Fwd Pkts,URG Flag Cnt
0,0.0,8569.50000,655.432936,49.510203,0.0,0,141385,1.0,19069.116850,113.166178,...,87.534438,20,0.0,119,8192,0.0,474.712955,1,553.0,0
1,0.0,0.00000,0.000000,3558.718861,0.0,0,281,17.0,174.655375,10676.156580,...,26.870058,20,0.0,0,123,0.0,21.939310,0,38.0,0
2,0.0,18494.57143,636.314186,53.605123,1.0,0,279824,1.0,24379.448340,92.915547,...,129.392497,20,0.0,1047,8192,0.0,566.234209,1,1086.0,0
3,0.0,0.00000,0.000000,0.000000,0.0,0,132,132.0,0.000000,15151.515150,...,0.000000,20,0.0,-1,256,0.0,0.000000,0,0.0,0
4,0.0,21082.83333,611.180489,47.442485,1.0,0,274016,1.0,26311.627030,80.287282,...,183.887722,20,0.0,1047,8192,0.0,497.254764,1,1285.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10819037,0.0,0.00000,0.000000,50000.000000,1.0,0,20,20.0,0.000000,100000.000000,...,0.000000,24,0.0,0,65535,0.0,0.000000,0,0.0,0
10819038,0.0,0.00000,0.000000,9.483617,0.0,0,105445,21.0,74531.176060,28.450851,...,0.000000,20,0.0,0,1024,0.0,0.000000,0,0.0,0
10819039,0.0,733857.00000,0.000000,2.725241,1.0,0,733880,22.0,423666.844400,5.450482,...,0.000000,32,0.0,0,8192,0.0,0.000000,0,0.0,0
10819040,0.0,732708.00000,0.000000,2.729526,1.0,0,732728,20.0,422996.540800,5.459052,...,0.000000,32,0.0,0,8192,0.0,0.000000,0,0.0,0


In [11]:
y

0                  Benign
1                  Benign
2                  Benign
3                  Benign
4                  Benign
                ...      
10819037    Infilteration
10819038           Benign
10819039           Benign
10819040           Benign
10819041    Infilteration
Name: Label, Length: 10819042, dtype: str

In [12]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded,
                                   random_state=104, 
                                   test_size=0.25, 
                                   shuffle=True)

In [15]:
params = {
    'device': 'cuda',
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss',

    'n_estimators': 300,
    'max_depth': 4,

    'learning_rate': 0.05,

    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'colsample_bylevel': 0.8,

    'min_child_weight': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 1.5,

    'tree_method': 'hist',
}

In [16]:
model = xgb.XGBClassifier(**params)

In [17]:
%%time

model.fit(X_train, y_train)

CPU times: user 5min 46s, sys: 5.44 s, total: 5min 51s
Wall time: 5min 26s


,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,0.8
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklear

In [18]:
y_preds = model.predict(X_test)

/home/kostas/Documents/git/NIDS_EDA/CSE_CIC_IDS2018/.venv/lib/python3.13/site-packages/xgboost/core.py:569: UserWarning: [21:58:15] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [21]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_preds))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99   2373757
           1       1.00      1.00      1.00     35962
           2       0.88      0.58      0.70       158
           3       0.90      0.86      0.88        51
           4       1.00      1.00      1.00     49591
           5       0.84      0.94      0.89       446
           6       1.00      1.00      1.00    143619
           7       1.00      1.00      1.00     10308
           8       1.00      1.00      1.00     36420
           9       0.28      0.42      0.33        12
          10       0.98      0.92      0.95      2444
          11       0.12      0.07      0.09        14
          12       0.55      0.01      0.02     28496
          13       1.00      0.39      0.56        23
          14       1.00      1.00      1.00     23460

    accuracy                           0.99   2704761
   macro avg       0.84      0.75      0.76   2704761
weighted avg       0.98   